# Embeddings & Vector Retrieval

This notebook builds the retrieval layer for the LLM system by generating
vector embeddings for customer feedback chunks and enabling similarity-based
search to retrieve relevant evidence.

In [1]:
import pandas as pd
import numpy as np

## Load Chunked Customer Feedback

We load pre-processed text chunks that will be embedded and indexed
for similarity-based retrieval.

In [2]:
chunked_df = pd.read_csv("../data/processed/review_chunks.csv")
chunked_df.head()

,text_chunk,sentiment
0,having tried a couple of other brands of glute...,positive
1,my cat loves these treats. if ever i can't fin...,positive
2,"first there was frosted mini-wheats, in origin...",negative
3,"fiber, 12g of sugar, 5g of protein and 200mg o...",negative
4,and i want to congratulate the graphic artist ...,positive


## Embedding Strategy

We convert each text chunk into a dense numerical vector that captures
semantic meaning, enabling similarity-based retrieval.

In [3]:
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 29.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 33.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 14.8 MB/s  0:00:25m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 22.6 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 38.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 39.7 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 30.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 16.2 MB/s  0:00:21m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 39.6 MB/s  0:00:04m0:00:0

In [4]:
from sentence_transformers import SentenceTransformer

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2")

## Generate Embeddings for Text Chunks

Each chunk is converted into a fixed-length embedding vector.

In [6]:
texts = chunked_df["text_chunk"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64
)

embeddings.shape

Batches: 100%|██████████| 141/141 [06:20<00:00,  2.70s/it]


(9010, 384)

## Build Vector Index

We use a simple cosine similarity-based index for retrieving
relevant chunks efficiently.

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
def retrieve_similar_chunks(query, top_k=5):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    return chunked_df.iloc[top_indices][["text_chunk", "sentiment"]]

## Test Retrieval with Sample Queries

In [9]:
query = "Customers are complaining about pricing and billing issues"
retrieve_similar_chunks(query)

,text_chunk,sentiment
3860,i have a standing order with amazon for some t...,positive
2494,"well, i bought 8 diff. items and all of them a...",positive
6937,i had already done the math before looking at ...,positive
4660,boo amazon! i thought i loved you for the conv...,negative
8566,love the cereal! but a word to the wise regard...,positive


## Retrieval Observations

- Retrieved chunks are semantically aligned with the query intent.
- Metadata such as sentiment can be used to filter results.
- Retrieval quality is critical for grounding LLM responses.

## Save Embeddings for Reuse

Saving embeddings avoids recomputation during experimentation.

In [10]:
np.save("../data/processed/review_embeddings.npy", embeddings)